# 🧠 NLP Fundamentals, Embeddings & Sentiment Analysis
### Deep Learning with TensorFlow / Keras — IMDB Movie Reviews

---

| Topic | Tools |
|-------|-------|
| Text Preprocessing | Pure Python, re, sklearn |
| Text Representation | CountVectorizer, TF-IDF |
| Word Embeddings | Keras Embedding layer, PCA |
| Sentiment Analysis | TF/Keras — IMDB dataset |
| Models | Dense · LSTM · Conv1D |

**Kernel:** Python (TensorFlow) — `tf_env`  
**Author:** TCHAPGA TCHITO C. | University of Buea — COT

---

## 📋 Table of Contents
1. [Introduction to NLP](#1)
2. [Text Preprocessing Pipeline](#2)
3. [Bag-of-Words & TF-IDF](#3)
4. [Word Embeddings — Theory](#4)
5. [Keras Tokenizer & Sequences](#5)
6. [Embedding Layer & Visualization](#6)
7. [IMDB Dataset — EDA](#7)
8. [Model 1 — Dense Network (BoW)](#8)
9. [Model 2 — Embedding + Bidirectional LSTM](#9)
10. [Model 3 — Embedding + Conv1D](#10)
11. [Model Comparison & Evaluation](#11)
12. [Live Inference on Custom Sentences](#12)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Global Imports & Settings
# ─────────────────────────────────────────────────────────────────────────────
import os, re, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Dense, Embedding, LSTM, Bidirectional, Conv1D,
    GlobalMaxPooling1D, Dropout, Flatten, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Pretty plotting
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print("✅ All imports successful")

---
<a id='1'></a>
## 1 · Introduction to Natural Language Processing

**Natural Language Processing (NLP)** is the branch of AI that enables computers to understand, interpret, and generate human language.

### Why NLP is Hard
- **Ambiguity** — *"I saw the man with the telescope"* (who has the telescope?)
- **Context** — meaning depends on surrounding words
- **Variability** — the same meaning expressed in countless ways
- **Informal text** — abbreviations, slang, emojis, typos

### Classic NLP Pipeline
```
Raw Text → Preprocessing → Representation → Model → Prediction
```

| Stage | Examples |
|-------|----------|
| **Preprocessing** | Tokenization, lowercasing, stop-word removal, stemming |
| **Representation** | Bag-of-Words, TF-IDF, Word Embeddings, Transformers |
| **Tasks** | Classification, NER, Translation, Q&A, Generation |

### Applications We'll Touch
✅ Sentiment Analysis (positive / negative)  
✅ Text vectorisation strategies  
✅ Dense word representations (embeddings)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — NLP Concept Illustration
# ─────────────────────────────────────────────────────────────────────────────
sentences = [
    "The movie was absolutely fantastic!",
    "I hated every single minute of this film.",
    "It was okay, nothing special.",
    "Best film I have ever seen in my entire life!",
    "Terrible acting and a boring story."
]

labels = ["😊 Positive", "😞 Negative", "😐 Neutral", "😊 Positive", "😞 Negative"]

print("─" * 65)
print(f"{'Review':<45} {'Label':<15}")
print("─" * 65)
for s, l in zip(sentences, labels):
    print(f"{s[:44]:<45} {l}")
print("─" * 65)
print("\n🎯 Goal: Train a model to predict the label automatically.")

---
<a id='2'></a>
## 2 · Text Preprocessing Pipeline

Before feeding text to a model, we must normalise and clean it.

| Step | Purpose | Example |
|------|---------|--------|
| **Lowercase** | Reduce vocabulary | `Movie → movie` |
| **Remove punctuation** | Eliminate noise | `great!!! → great` |
| **Tokenisation** | Split into words/tokens | `I love NLP → ['I','love','NLP']` |
| **Stop-word removal** | Drop very common words | `the, a, is, of …` |
| **Stemming** | Reduce to word root (crude) | `running → run` |
| **Lemmatisation** | Morphological root (precise) | `better → good` |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Text Preprocessing (no external NLP library needed)
# ─────────────────────────────────────────────────────────────────────────────

# A minimal English stop-word list
STOP_WORDS = {
    'a','an','the','and','or','but','is','are','was','were','be','been',
    'being','have','has','had','do','does','did','will','would','could',
    'should','may','might','shall','can','i','me','my','we','our','you',
    'your','he','she','it','they','them','his','her','its','their','this',
    'that','these','those','of','in','on','at','to','for','by','with',
    'from','as','not','no','so','if','about','what','which','who','how'
}

# A minimal rule-based stemmer (suffix stripping)
def simple_stem(word):
    suffixes = ['ing','tion','ness','ful','less','ed','er','est','ly','ity']
    for suf in suffixes:
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[:-len(suf)]
    return word

def preprocess(text, remove_stops=True, stem=False):
    """Full preprocessing pipeline."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 3. Keep only alphabetic characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 4. Tokenise
    tokens = text.split()
    # 5. Remove stop words
    if remove_stops:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    # 6. Stemming (optional)
    if stem:
        tokens = [simple_stem(t) for t in tokens]
    # 7. Remove very short tokens
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

# ── Demo ──────────────────────────────────────────────────────────────────────
demo_text = "The acting was BRILLIANT! Absolutely loved the film, especially the ending."

print("Original  :", demo_text)
print("Lowercase :", demo_text.lower())
print("Tokens    :", preprocess(demo_text, remove_stops=False, stem=False))
print("No stops  :", preprocess(demo_text, remove_stops=True,  stem=False))
print("Stemmed   :", preprocess(demo_text, remove_stops=True,  stem=True))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Visualise Vocabulary Reduction After Preprocessing
# ─────────────────────────────────────────────────────────────────────────────
corpus = [
    "The movie was absolutely fantastic and brilliantly directed!",
    "I hated every single minute of this terrible film.",
    "The actors were amazing and the story was very engaging.",
    "Boring, predictable, and absolutely dreadful acting.",
    "A masterpiece of storytelling and emotional depth.",
    "The worst film I have ever watched in my entire life!",
    "Loved every moment — beautiful cinematography and music.",
    "Slow, confusing, and disappointing from start to finish."
]

raw_vocab  = set(w.lower().strip(string.punctuation) for t in corpus for w in t.split() if w)
proc_vocab = set(tok for t in corpus for tok in preprocess(t, remove_stops=True, stem=True))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
stages = ['Raw tokens', 'After preprocessing']
counts = [len(raw_vocab), len(proc_vocab)]
colors = ['#5B8DBE', '#2E8B57']

for ax, stage, count, color in zip(axes, stages, counts, colors):
    ax.bar([stage], [count], color=color, width=0.4, edgecolor='white')
    ax.set_ylim(0, max(counts) * 1.3)
    ax.set_ylabel('Unique tokens')
    ax.set_title(stage, fontweight='bold')
    ax.text(0, count + 0.5, str(count), ha='center', fontsize=14, fontweight='bold')

reduction = 100 * (1 - proc_vocab.__len__() / raw_vocab.__len__())
fig.suptitle(f'Vocabulary Reduction after Preprocessing  ({reduction:.0f}% smaller)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f"Raw vocab  : {sorted(raw_vocab)[:12]} …")
print(f"Clean vocab: {sorted(proc_vocab)[:12]} …")

---
<a id='3'></a>
## 3 · Bag-of-Words & TF-IDF

These are the two classical ways to convert text into **fixed-length numerical vectors**.

### 3.1 Bag-of-Words (BoW)
Each document is represented as a vector of **word counts**.

```
Vocab  : [cat, dog, film, great, hate, love, movie]
"I love this movie"  → [0, 0, 0, 0, 0, 1, 1]
"I hate this movie"  → [0, 0, 0, 0, 1, 0, 1]
```

**Pros:** Simple, interpretable  
**Cons:** Ignores word order; very high-dimensional sparse vectors

### 3.2 TF-IDF (Term Frequency – Inverse Document Frequency)
Weights words by how **informative** they are across the corpus.

$$\text{TF-IDF}(t, d) = \underbrace{\frac{\text{count}(t,d)}{|d|}}_{\text{TF}} \times \underbrace{\log\frac{N}{1 + df(t)}}_{\text{IDF}}$$

- Common words (*the, a*) get low IDF → low score
- Rare, topic-specific words (*cinematic, masterpiece*) get high score

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Bag-of-Words with CountVectorizer
# ─────────────────────────────────────────────────────────────────────────────
corpus_small = [
    "I love this movie so much",
    "This movie is terrible I hate it",
    "Great film wonderful acting great story",
    "Boring film bad acting and bad story",
    "Absolutely loved the film beautiful music"
]
label_small = [1, 0, 1, 0, 1]  # 1=positive, 0=negative

# Fit BoW
cv = CountVectorizer(max_features=15)
X_bow = cv.fit_transform(corpus_small).toarray()
vocab = cv.get_feature_names_out()

print("Vocabulary:", list(vocab))
print(f"\nBoW matrix shape: {X_bow.shape}  ({len(corpus_small)} docs × {len(vocab)} words)\n")

df_bow = pd.DataFrame(X_bow, columns=vocab)
df_bow.index = [f"Doc {i+1}" for i in range(len(corpus_small))]

# Heatmap
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(df_bow, annot=True, fmt='d', cmap='Blues', linewidths=0.5,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_title('Bag-of-Words Matrix', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Words in Vocabulary')
plt.tight_layout()
plt.show()
print("\nSample reviews:")
for i, (s, l) in enumerate(zip(corpus_small, label_small)):
    print(f"  Doc {i+1} ({'POS' if l else 'NEG'}): {s}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — TF-IDF Vectoriser
# ─────────────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=15, use_idf=True)
X_tfidf = tfidf.fit_transform(corpus_small).toarray()
vocab_tf = tfidf.get_feature_names_out()

df_tfidf = pd.DataFrame(np.round(X_tfidf, 3), columns=vocab_tf)
df_tfidf.index = [f"Doc {i+1}" for i in range(len(corpus_small))]

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

# BoW
sns.heatmap(df_bow, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            linewidths=0.4, cbar=False)
axes[0].set_title('Bag-of-Words (raw counts)', fontweight='bold')

# TF-IDF
sns.heatmap(df_tfidf, annot=True, fmt='.2f', cmap='Greens', ax=axes[1],
            linewidths=0.4, cbar=False)
axes[1].set_title('TF-IDF (weighted scores)', fontweight='bold')

plt.suptitle('BoW vs TF-IDF — same corpus', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Top TF-IDF words per document
print("\nTop-3 TF-IDF words per document:")
for i in range(len(corpus_small)):
    top3_idx = X_tfidf[i].argsort()[::-1][:3]
    top3 = [(vocab_tf[j], round(X_tfidf[i,j], 3)) for j in top3_idx if X_tfidf[i,j] > 0]
    print(f"  Doc {i+1}: {top3}")

---
<a id='4'></a>
## 4 · Word Embeddings — Theory & Intuition

BoW and TF-IDF suffer from:
- **Sparsity** — vectors are mostly zeros (huge vocabulary)
- **No semantics** — *"great"* and *"fantastic"* look completely different
- **No context** — word order is ignored

### The Embedding Idea
Map each word to a **dense, low-dimensional real-valued vector**:

```
"king"    → [0.82,  0.41, -0.12, …]  ← 300 dimensions
"queen"   → [0.79,  0.45, -0.10, …]  ← very close!
"film"    → [-0.3,  0.88,  0.52, …]  ← far away
```

### Distributional Hypothesis
> *"You shall know a word by the company it keeps"* — J.R. Firth (1957)

Words appearing in **similar contexts** get **similar vectors**.

### Word2Vec: Two Architectures
| Architecture | Input → Predict |
|---|---|
| **CBOW** | Context words → Centre word |
| **Skip-gram** | Centre word → Context words |

### Famous Semantic Arithmetic
$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

### Modern Alternatives
- **GloVe** (Global Vectors) — matrix factorisation on co-occurrence
- **FastText** — character n-grams (handles unknown/misspelled words)
- **BERT / GPT** — contextual embeddings (same word ≠ same vector)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Embedding Intuition: Sparse vs Dense Representation
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ─ Sparse BoW ─────────────────────────────────────────────────────────────
vocab_demo = ['great','terrible','film','acting','music','love','hate',
              'boring','brilliant','awful']
words_demo = ['great', 'brilliant', 'terrible']
sparse = np.eye(len(vocab_demo))[[vocab_demo.index(w) for w in words_demo]]

im0 = axes[0].imshow(sparse, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[0].set_xticks(range(len(vocab_demo)))
axes[0].set_xticklabels(vocab_demo, rotation=40, ha='right', fontsize=9)
axes[0].set_yticks(range(len(words_demo)))
axes[0].set_yticklabels(words_demo, fontsize=10)
axes[0].set_title('Sparse One-Hot / BoW\n(vocabulary-sized, mostly zeros)',
                   fontweight='bold')
axes[0].set_xlabel('Vocabulary dimension (10 shown, real vocab ~50 000)')

for i in range(len(words_demo)):
    for j in range(len(vocab_demo)):
        axes[0].text(j, i, str(int(sparse[i, j])), ha='center', va='center',
                     color='white' if sparse[i,j] > 0.5 else 'black', fontsize=9)

# ─ Dense Embedding ─────────────────────────────────────────────────────────
np.random.seed(7)
dense = np.random.randn(3, 8)
# Make great/brilliant similar, terrible different
dense[1] = dense[0] + np.random.randn(8) * 0.2
dense[2] = -dense[0] + np.random.randn(8) * 0.2

im1 = axes[1].imshow(dense, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)
axes[1].set_xticks(range(8))
axes[1].set_xticklabels([f'd{i+1}' for i in range(8)], fontsize=9)
axes[1].set_yticks(range(3))
axes[1].set_yticklabels(words_demo, fontsize=10)
axes[1].set_title('Dense Word Embedding\n(low-dimensional, semantically similar = similar rows)',
                   fontweight='bold')
axes[1].set_xlabel('Embedding dimension (8 shown, typical: 64–512)')

for i in range(3):
    for j in range(8):
        axes[1].text(j, i, f"{dense[i,j]:.2f}", ha='center', va='center', fontsize=8)

plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.suptitle('Sparse vs Dense Text Representations', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Cosine similarity demo
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nCosine similarities (dense vectors):")
print(f"  great  ↔  brilliant : {cosine_sim(dense[0], dense[1]):.3f}  ← HIGH (synonyms)")
print(f"  great  ↔  terrible  : {cosine_sim(dense[0], dense[2]):.3f}  ← LOW  (antonyms)")
print(f"  brilliant ↔ terrible: {cosine_sim(dense[1], dense[2]):.3f}  ← LOW")

---
<a id='5'></a>
## 5 · Keras Tokenizer & Sequence Padding

Before using an Embedding layer, we need to:
1. **Build a vocabulary** — assign an integer index to each unique word
2. **Encode sentences** — replace each word with its index
3. **Pad sequences** — make all sequences the same length

```
"I love this film"  →  [3, 7, 12, 5]
"Terrible"          →  [9]
After padding →  [[3, 7, 12, 5], [0, 0, 0, 9]]
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Keras Tokenizer & Padding
# ─────────────────────────────────────────────────────────────────────────────
sample_texts = [
    "The film was absolutely brilliant",
    "Terrible and boring movie",
    "I loved every single moment of the film",
    "Worst acting I have ever seen",
    "A beautiful cinematic masterpiece"
]

VOCAB_SIZE  = 50    # keep top-50 words
OOV_TOKEN   = '<OOV>'  # out-of-vocabulary token
MAX_LEN     = 10    # pad/truncate to this length

# 1. Fit tokeniser
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(sample_texts)

word_index = tokenizer.word_index
print(f"Vocabulary size (word_index): {len(word_index)} words")
print(f"First 15 entries: {dict(list(word_index.items())[:15])}")

# 2. Convert texts → sequences of integers
sequences = tokenizer.texts_to_sequences(sample_texts)
print("\nEncoded sequences (before padding):")
for txt, seq in zip(sample_texts, sequences):
    print(f"  {seq}  ← '{txt}'")

# 3. Pad sequences (post-padding with 0)
padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
print(f"\nPadded matrix (shape {padded.shape}):")
df_pad = pd.DataFrame(padded, columns=[f'pos{i+1}' for i in range(MAX_LEN)])
df_pad.index = [f'S{i+1}' for i in range(len(sample_texts))]
print(df_pad.to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — Visualise Padding
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
mask = (padded > 0).astype(int)  # 1 = real token, 0 = padding

sns.heatmap(padded, annot=True, fmt='d', cmap='Blues', linewidths=0.6,
            ax=ax, cbar_kws={'label': 'Token ID'})
ax.set_xticklabels([f'pos {i+1}' for i in range(MAX_LEN)], rotation=0)
ax.set_yticklabels([f'S{i+1}: "{s[:25]}"' for i, s in enumerate(sample_texts)],
                    rotation=0, fontsize=8.5)
ax.set_title('Padded Sequences (0 = padding)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

# OOV demo
new_sentences = ["Incredible acting and breathtaking visuals",  # 'Incredible' is OOV
                 "The film was lovely"]
new_seq = tokenizer.texts_to_sequences(new_sentences)
new_pad = pad_sequences(new_seq, maxlen=MAX_LEN, padding='post')

print("\nOOV Demo (unseen words → token 1):")
for s, seq in zip(new_sentences, new_seq):
    print(f"  '{s}'")
    print(f"  → {seq}  (1 = OOV token '<OOV>')")

---
<a id='6'></a>
## 6 · Embedding Layer in Keras & Visualisation

The `Embedding(vocab_size, embed_dim)` layer:
- Acts as a **lookup table** — each word index retrieves a learned vector
- Weights are **randomly initialised** then updated by backpropagation
- Input: `(batch, sequence_length)` → Output: `(batch, sequence_length, embed_dim)`

```
Embedding(10000, 64)  →  10 000 words × 64 dimensions  =  640 000 trainable parameters
```

After training, similar words cluster together in embedding space — we can see this with **PCA** or **t-SNE**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 — Build a Tiny Embedding Model & Extract Vectors
# ─────────────────────────────────────────────────────────────────────────────
# Mini sentiment dataset for demo
mini_texts  = [
    "brilliant fantastic wonderful amazing",
    "terrible awful dreadful horrible",
    "great excellent superb outstanding",
    "bad dull boring mediocre",
    "loved enjoyed appreciated adored",
    "hated despised loathed disliked"
]
mini_labels = [1, 0, 1, 0, 1, 0]  # 1=pos, 0=neg

# Tokenise
tok_mini = Tokenizer(oov_token='<OOV>')
tok_mini.fit_on_texts(mini_texts)
V = len(tok_mini.word_index) + 1   # vocab size
EMBED_DIM = 4

X_mini = pad_sequences(tok_mini.texts_to_sequences(mini_texts), maxlen=4, padding='post')
y_mini = np.array(mini_labels)

# Small model just to show embedding extraction
embed_model = Sequential([
    Embedding(V, EMBED_DIM, input_length=4, name='embedding'),
    GlobalAveragePooling1D(),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])
embed_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
embed_model.summary()

# Train briefly
embed_model.fit(X_mini, y_mini, epochs=50, verbose=0)
train_acc = embed_model.evaluate(X_mini, y_mini, verbose=0)[1]
print(f"\nTraining accuracy on mini dataset: {train_acc:.2%}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 — Visualise Learned Embeddings with PCA
# ─────────────────────────────────────────────────────────────────────────────
# Extract embedding weights
emb_weights = embed_model.get_layer('embedding').get_weights()[0]  # shape: (V, EMBED_DIM)
idx_to_word = {v: k for k, v in tok_mini.word_index.items()}

# Project to 2D with PCA
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(emb_weights[1:])  # skip index 0 (padding)
words_list = [idx_to_word[i] for i in range(1, len(idx_to_word) + 1)]

# Colour by sentiment
positive_words = {'brilliant','fantastic','wonderful','amazing','great','excellent',
                  'superb','outstanding','loved','enjoyed','appreciated','adored'}
colors_emb = ['#2E8B57' if w in positive_words else '#CC3333' for w in words_list]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(coords_2d[:, 0], coords_2d[:, 1],
           c=colors_emb, s=80, alpha=0.8, zorder=3)

for i, (word, c) in enumerate(zip(words_list, colors_emb)):
    ax.annotate(word, (coords_2d[i, 0], coords_2d[i, 1]),
                textcoords='offset points', xytext=(5, 5),
                fontsize=9, color=c)

# Legend
from matplotlib.patches import Patch
legend = [Patch(facecolor='#2E8B57', label='Positive words'),
          Patch(facecolor='#CC3333', label='Negative words')]
ax.legend(handles=legend, fontsize=10)
ax.axhline(0, color='lightgrey', lw=0.8)
ax.axvline(0, color='lightgrey', lw=0.8)
ax.set_title(f'Learned Embedding Space (PCA 2D projection)\n'
             f'Green=Positive, Red=Negative — trained on mini dataset',
             fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()
print("\nObservation: positive words cluster together; negative words cluster separately.")
print("With more training data & larger embeddings, clusters become cleaner.")

---
<a id='7'></a>
## 7 · IMDB Dataset — Exploratory Data Analysis

The **IMDB Large Movie Review Dataset** is a standard NLP benchmark:

| Property | Value |
|---|---|
| Total reviews | 50 000 |
| Training set | 25 000 |
| Test set | 25 000 |
| Classes | Positive (1) / Negative (0) |
| Class balance | Perfectly balanced (50/50) |
| Vocabulary | 88 584 unique words |
| Pre-processing | Each review stored as a list of word indices |

> **Source:** Andrew L. Maas et al., *"Learning Word Vectors for Sentiment Analysis"*, ACL 2011.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 19 — Load IMDB Dataset
# ─────────────────────────────────────────────────────────────────────────────
NUM_WORDS   = 10000   # keep only top 10 000 words
INDEX_SHIFT = 3       # 0=padding, 1=start, 2=unknown, 3=unused

(x_train_raw, y_train), (x_test_raw, y_test) = imdb.load_data(num_words=NUM_WORDS)

print(f"Training reviews : {len(x_train_raw):>6} | labels: {y_train.shape}")
print(f"Test reviews     : {len(x_test_raw):>6} | labels: {y_test.shape}")
print(f"\nClass distribution (train):")
print(f"  Positive (1): {sum(y_train):>5}  ({sum(y_train)/len(y_train)*100:.0f}%)")
print(f"  Negative (0): {sum(1-y_train):>5}  ({sum(1-y_train)/len(y_train)*100:.0f}%)")

# Decode a sample review back to text
word_index = imdb.get_word_index()
reverse_index = {v + INDEX_SHIFT: k for k, v in word_index.items()}
reverse_index[0] = '<PAD>'
reverse_index[1] = '<START>'
reverse_index[2] = '<UNK>'

def decode_review(seq):
    return ' '.join(reverse_index.get(i, '?') for i in seq)

print("\n--- Sample positive review (truncated) ---")
print(decode_review(x_train_raw[0])[:500], "...")
print(f"Label: {'POSITIVE ✅' if y_train[0] else 'NEGATIVE ❌'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 20 — EDA: Review Length Distribution
# ─────────────────────────────────────────────────────────────────────────────
train_lengths = [len(r) for r in x_train_raw]
test_lengths  = [len(r) for r in x_test_raw]

print("Review length statistics (train):")
print(f"  Min    : {min(train_lengths)}")
print(f"  Max    : {max(train_lengths)}")
print(f"  Mean   : {np.mean(train_lengths):.0f}")
print(f"  Median : {np.median(train_lengths):.0f}")
print(f"  90th % : {np.percentile(train_lengths, 90):.0f}")
print(f"  95th % : {np.percentile(train_lengths, 95):.0f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution
axes[0].hist(train_lengths, bins=80, color='#3A7EBF', edgecolor='white', alpha=0.85)
axes[0].axvline(np.median(train_lengths), color='orange', lw=2, label=f'Median={np.median(train_lengths):.0f}')
axes[0].axvline(np.percentile(train_lengths, 95), color='red', lw=1.5, ls='--',
                label=f'95th pct={np.percentile(train_lengths,95):.0f}')
axes[0].set_xlabel('Review length (tokens)')
axes[0].set_ylabel('Count')
axes[0].set_title('Review Length Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# Class balance
axes[1].bar(['Positive', 'Negative'], [sum(y_train), sum(1-y_train)],
            color=['#2E8B57', '#CC3333'], edgecolor='white', width=0.5)
axes[1].set_title('Class Balance (Train)', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate([sum(y_train), sum(1-y_train)]):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Length by class
pos_lens = [train_lengths[i] for i in range(len(y_train)) if y_train[i] == 1]
neg_lens = [train_lengths[i] for i in range(len(y_train)) if y_train[i] == 0]
axes[2].hist(pos_lens, bins=60, alpha=0.6, color='#2E8B57', label='Positive', edgecolor='none')
axes[2].hist(neg_lens, bins=60, alpha=0.6, color='#CC3333', label='Negative', edgecolor='none')
axes[2].set_xlabel('Review length (tokens)')
axes[2].set_title('Length by Sentiment', fontweight='bold')
axes[2].legend()

plt.suptitle('IMDB Dataset — Exploratory Data Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 21 — Prepare Data for All Models
# ─────────────────────────────────────────────────────────────────────────────
MAXLEN = 200    # keep/pad to 200 tokens  (covers ~85% of reviews fully)

# ── Padded sequences (for Embedding-based models) ────────────────────────────
x_train_pad = pad_sequences(x_train_raw, maxlen=MAXLEN, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test_raw,  maxlen=MAXLEN, padding='post', truncating='post')

print(f"x_train_pad shape : {x_train_pad.shape}")
print(f"x_test_pad  shape : {x_test_pad.shape}")

# ── Multi-hot BoW vectors (for Dense model) ────────────────────────────────
def multi_hot_encode(sequences, vocab_size):
    """Convert list of sequences into multi-hot binary matrix."""
    result = np.zeros((len(sequences), vocab_size), dtype='float32')
    for i, seq in enumerate(sequences):
        result[i, seq] = 1.0
    return result

x_train_mh = multi_hot_encode(x_train_raw, NUM_WORDS)
x_test_mh  = multi_hot_encode(x_test_raw,  NUM_WORDS)

print(f"\nx_train_mh shape  : {x_train_mh.shape}  (multi-hot BoW)")
print(f"x_test_mh  shape  : {x_test_mh.shape}")
print(f"\nData ready for model training!")
print(f"  Vocab size (NUM_WORDS) : {NUM_WORDS}")
print(f"  Max sequence length    : {MAXLEN}")

---
<a id='8'></a>
## 8 · Model 1 — Dense Network on Bag-of-Words

**Architecture:** Multi-Hot BoW vector → Dense → Dense → Sigmoid

```
Input: 10 000-dim binary vector (1 = word present)
  ↓  Dense(64, relu)
  ↓  Dropout(0.3)
  ↓  Dense(32, relu)
  ↓  Dense(1, sigmoid)
Output: probability of POSITIVE sentiment
```

**Pros:** Fast to train, strong baseline  
**Cons:** Ignores word order completely

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 23 — Model 1: Dense Network (BoW)
# ─────────────────────────────────────────────────────────────────────────────
model1 = Sequential([
    Dense(64,  activation='relu', input_shape=(NUM_WORDS,)),
    Dropout(0.3),
    Dense(32,  activation='relu'),
    Dropout(0.2),
    Dense(1,   activation='sigmoid')
], name='Dense_BoW')

model1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model1.summary()

# Train
es = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history1 = model1.fit(
    x_train_mh, y_train,
    epochs=15,
    batch_size=512,
    validation_split=0.1,
    callbacks=[es],
    verbose=1
)

# Evaluate
loss1, acc1 = model1.evaluate(x_test_mh, y_test, verbose=0)
print(f"\n📊 Model 1 Test Accuracy : {acc1:.4f}  ({acc1*100:.2f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 24 — Plot Training History (Model 1)
# ─────────────────────────────────────────────────────────────────────────────
def plot_history(history, title, color='steelblue', ax_pair=None):
    """Plot accuracy and loss curves side by side."""
    if ax_pair is None:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    else:
        axes = ax_pair
    
    epochs = range(1, len(history.history['accuracy']) + 1)
    
    # Accuracy
    axes[0].plot(epochs, history.history['accuracy'],     color=color,   lw=2, label='Train')
    axes[0].plot(epochs, history.history['val_accuracy'], color=color,   lw=2, ls='--', label='Validation')
    axes[0].set_title(f'{title} — Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].set_ylim(0.5, 1.02)
    axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    
    # Loss
    axes[1].plot(epochs, history.history['loss'],     color=color, lw=2, label='Train')
    axes[1].plot(epochs, history.history['val_loss'], color=color, lw=2, ls='--', label='Validation')
    axes[1].set_title(f'{title} — Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Binary CrossEntropy')
    axes[1].legend()
    
    if ax_pair is None:
        plt.tight_layout()
        plt.show()

plot_history(history1, 'Model 1 — Dense / BoW', color='#2E6DA4')

---
<a id='9'></a>
## 9 · Model 2 — Embedding + Bidirectional LSTM

**LSTM** (Long Short-Term Memory) is a recurrent network that captures **sequential dependencies** — crucial for language.

**Bidirectional LSTM** processes the sequence **both forwards and backwards**, giving the model both past and future context at each step.

```
Input: (batch, 200)  → integer token IDs
  ↓  Embedding(10000, 64)
  ↓  Bidirectional(LSTM(64))  →  output: (batch, 128)
  ↓  Dense(64, relu)
  ↓  Dropout(0.3)
  ↓  Dense(1, sigmoid)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 26 — Model 2: Embedding + Bidirectional LSTM
# ─────────────────────────────────────────────────────────────────────────────
EMBED_DIM_M2 = 64
LSTM_UNITS   = 64

model2 = Sequential([
    Embedding(NUM_WORDS, EMBED_DIM_M2, input_length=MAXLEN, name='embed_lstm'),
    Bidirectional(LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='BiLSTM_Embedding')

model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model2.summary()
print(f"\nEmbedding params: {NUM_WORDS * EMBED_DIM_M2:,}  ({NUM_WORDS} words × {EMBED_DIM_M2} dims)")

# Train
es2 = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history2 = model2.fit(
    x_train_pad, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    callbacks=[es2],
    verbose=1
)

loss2, acc2 = model2.evaluate(x_test_pad, y_test, verbose=0)
print(f"\n📊 Model 2 Test Accuracy : {acc2:.4f}  ({acc2*100:.2f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 27 — Plot Training History (Model 2)
# ─────────────────────────────────────────────────────────────────────────────
plot_history(history2, 'Model 2 — Embedding + BiLSTM', color='#9B2C2C')

---
<a id='10'></a>
## 10 · Model 3 — Embedding + Conv1D

**Conv1D** applies 1-D convolutional filters that slide over the sequence — excellent at capturing **local n-gram patterns** (e.g., *"not good"*, *"absolutely loved"*).

**GlobalMaxPooling1D** selects the strongest feature across the entire sequence.

```
Input: (batch, 200)  → integer token IDs
  ↓  Embedding(10000, 64)
  ↓  Conv1D(128 filters, kernel=5, relu)  → captures 5-grams
  ↓  GlobalMaxPooling1D()                 → take the max feature
  ↓  Dense(64, relu)
  ↓  Dropout(0.3)
  ↓  Dense(1, sigmoid)
```

**Pros:** Faster than LSTM, captures local patterns well  
**Cons:** Less context than LSTM for long-range dependencies

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 29 — Model 3: Embedding + Conv1D + GlobalMaxPooling
# ─────────────────────────────────────────────────────────────────────────────
EMBED_DIM_M3 = 64
FILTERS      = 128
KERNEL_SIZE  = 5

model3 = Sequential([
    Embedding(NUM_WORDS, EMBED_DIM_M3, input_length=MAXLEN, name='embed_conv'),
    Conv1D(FILTERS, KERNEL_SIZE, activation='relu', padding='valid'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
], name='Conv1D_Embedding')

model3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model3.summary()

# Train
es3 = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history3 = model3.fit(
    x_train_pad, y_train,
    epochs=15,
    batch_size=256,
    validation_split=0.1,
    callbacks=[es3],
    verbose=1
)

loss3, acc3 = model3.evaluate(x_test_pad, y_test, verbose=0)
print(f"\n📊 Model 3 Test Accuracy : {acc3:.4f}  ({acc3*100:.2f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 30 — Plot Training History (Model 3)
# ─────────────────────────────────────────────────────────────────────────────
plot_history(history3, 'Model 3 — Embedding + Conv1D', color='#2E7D32')

---
<a id='11'></a>
## 11 · Model Comparison & Evaluation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 32 — Side-by-Side Training Curves + Summary Table
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Model 1\nDense/BoW': '#2E6DA4',
          'Model 2\nBiLSTM':    '#9B2C2C',
          'Model 3\nConv1D':    '#2E7D32'}
histories = [history1, history2, history3]

for (lbl, col), hist in zip(colors.items(), histories):
    n = len(hist.history['accuracy'])
    e = range(1, n + 1)
    axes[0].plot(e, hist.history['val_accuracy'], color=col, lw=2.5, label=lbl)
    axes[1].plot(e, hist.history['val_loss'],     color=col, lw=2.5, label=lbl)

axes[0].set_title('Validation Accuracy — All Models', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=9); axes[0].set_ylim(0.7, 1.0)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

axes[1].set_title('Validation Loss — All Models', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=9)

plt.suptitle('Training Comparison — IMDB Sentiment Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
results = pd.DataFrame({
    'Model': ['Dense + BoW', 'Bidirectional LSTM', 'Conv1D'],
    'Input Type': ['Multi-hot (10K)', 'Padded sequences (200)', 'Padded sequences (200)'],
    'Embedding': ['No', 'Yes (64-dim)', 'Yes (64-dim)'],
    'Test Accuracy': [f'{acc1*100:.2f}%', f'{acc2*100:.2f}%', f'{acc3*100:.2f}%'],
    'Test Loss': [f'{loss1:.4f}', f'{loss2:.4f}', f'{loss3:.4f}'],
    'Params': [f'{model1.count_params():,}', f'{model2.count_params():,}', f'{model3.count_params():,}']
})
print("\n" + "═" * 85)
print(" RESULTS SUMMARY")
print("═" * 85)
print(results.to_string(index=False))
print("═" * 85)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 33 — Confusion Matrices for All 3 Models
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
model_names = ['Model 1: Dense/BoW', 'Model 2: BiLSTM', 'Model 3: Conv1D']
model_list  = [model1, model2, model3]
x_inputs    = [x_test_mh, x_test_pad, x_test_pad]
accs        = [acc1, acc2, acc3]
palettes    = ['Blues', 'Reds', 'Greens']

for ax, name, model, x_in, acc, pal in zip(axes, model_names, model_list, x_inputs, accs, palettes):
    y_pred_prob = model.predict(x_in, verbose=0)
    y_pred      = (y_pred_prob > 0.5).astype(int).ravel()
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap=pal, ax=ax,
                xticklabels=['Predicted NEG', 'Predicted POS'],
                yticklabels=['Actual NEG', 'Actual POS'],
                cbar=False, linewidths=0.5)
    ax.set_title(f'{name}\nAcc: {acc*100:.2f}%', fontweight='bold', fontsize=10)

plt.suptitle('Confusion Matrices — IMDB Test Set (25 000 reviews)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Best model classification report
best_idx   = np.argmax([acc1, acc2, acc3])
best_model = model_list[best_idx]
best_x     = x_inputs[best_idx]
best_name  = model_names[best_idx]

y_pred_best = (best_model.predict(best_x, verbose=0) > 0.5).astype(int).ravel()
print(f"\nDetailed report — {best_name} (best model):")
print(classification_report(y_test, y_pred_best, target_names=['Negative', 'Positive']))

---
<a id='12'></a>
## 12 · Live Inference on Custom Sentences

Now let's use the trained models to predict sentiment on **our own text** — text never seen during training.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 35 — Inference on Custom Text
# ─────────────────────────────────────────────────────────────────────────────

def encode_text_for_models(text, word_index, num_words, maxlen):
    """Encode a raw text string into both multi-hot and padded sequence."""
    # Tokenise
    tokens = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    # Encode to integer sequence (shift by 3 to match IMDB encoding)
    seq = [word_index.get(w, 2) + 3 for w in tokens if word_index.get(w, 0) < num_words]
    # Padded sequence
    pad = pad_sequences([seq], maxlen=maxlen, padding='post', truncating='post')
    # Multi-hot
    mh = np.zeros((1, num_words), dtype='float32')
    for idx in seq:
        if idx < num_words:
            mh[0, idx] = 1.0
    return mh, pad


def predict_sentiment(text):
    """Predict sentiment with all three models and display a verdict."""
    mh, pad = encode_text_for_models(text, word_index, NUM_WORDS, MAXLEN)
    
    p1 = model1.predict(mh,  verbose=0)[0, 0]
    p2 = model2.predict(pad, verbose=0)[0, 0]
    p3 = model3.predict(pad, verbose=0)[0, 0]
    
    avg = (p1 + p2 + p3) / 3
    sentiment = '😊 POSITIVE' if avg > 0.5 else '😞 NEGATIVE'
    confidence = max(avg, 1 - avg) * 100
    
    print(f"\n📝 Review: \"{text[:80]}{'...' if len(text)>80 else ''}\"")
    print(f"   Dense/BoW score   : {p1:.3f}  {'🟢' if p1>0.5 else '🔴'}")
    print(f"   BiLSTM score      : {p2:.3f}  {'🟢' if p2>0.5 else '🔴'}")
    print(f"   Conv1D score      : {p3:.3f}  {'🟢' if p3>0.5 else '🔴'}")
    print(f"   Ensemble average  : {avg:.3f}")
    print(f"   ► Verdict : {sentiment}  (confidence: {confidence:.1f}%)")
    return avg


# Test sentences
test_reviews = [
    "This film was an absolute masterpiece. The acting was superb and the story deeply moving.",
    "Terrible film. Worst acting I have ever seen. Complete waste of time and money.",
    "An average film with some good moments but nothing particularly memorable.",
    "I loved every second of this movie. Brilliant performances by the entire cast!",
    "Boring, predictable, and painfully long. I almost fell asleep in the cinema.",
    "The cinematography was beautiful but the script was weak and the ending disappointing."
]

print("=" * 70)
print("         SENTIMENT ANALYSIS — CUSTOM REVIEWS")
print("=" * 70)
for review in test_reviews:
    predict_sentiment(review)
print("\n" + "=" * 70)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 36 — Visual Sentiment Bars
# ─────────────────────────────────────────────────────────────────────────────
def predict_scores(text):
    mh, pad = encode_text_for_models(text, word_index, NUM_WORDS, MAXLEN)
    p1 = float(model1.predict(mh,  verbose=0)[0, 0])
    p2 = float(model2.predict(pad, verbose=0)[0, 0])
    p3 = float(model3.predict(pad, verbose=0)[0, 0])
    return p1, p2, p3, (p1+p2+p3)/3

review_labels = [
    "Masterpiece — superb acting!",
    "Terrible — worst film ever.",
    "Average, nothing memorable.",
    "Loved every second! Brilliant!",
    "Boring, long, almost fell asleep.",
    "Beautiful photography, weak script."
]
all_scores = [predict_scores(r) for r in test_reviews]
ensemble   = [s[3] for s in all_scores]

fig, ax = plt.subplots(figsize=(11, 5))
y_pos = np.arange(len(review_labels))
bar_colors = ['#2E8B57' if s > 0.5 else '#CC3333' for s in ensemble]

bars = ax.barh(y_pos, ensemble, color=bar_colors, edgecolor='white', height=0.6)
ax.axvline(0.5, color='black', lw=1.5, ls='--', alpha=0.7, label='Decision boundary')
ax.set_yticks(y_pos)
ax.set_yticklabels(review_labels, fontsize=10)
ax.set_xlim(0, 1)
ax.set_xlabel('Ensemble Sentiment Score (0=Negative, 1=Positive)', fontsize=11)
ax.set_title('Sentiment Predictions — Ensemble of 3 Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)

for bar, score in zip(bars, ensemble):
    label = f"{score:.2f} {'😊' if score>0.5 else '😞'}"
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10)

plt.tight_layout()
plt.show()

---
## 📚 Summary & Key Takeaways

### What We Covered

| Concept | Key Insight |
|---|---|
| **Preprocessing** | Lowercasing, stop-word removal, and stemming reduce vocabulary by ~40-60% |
| **Bag-of-Words** | Fast and effective, but ignores word order and semantics |
| **TF-IDF** | Weights informative words higher; better for retrieval tasks |
| **Word Embeddings** | Dense 64-512 dim vectors; similar words cluster together |
| **Keras Tokenizer** | Converts text to integer sequences; OOV token handles unseen words |
| **Padding** | All sequences must be the same length for batch processing |
| **Embedding layer** | Learned lookup table; 64-dim × 10K vocab = 640K trainable params |
| **Dense + BoW** | Strong baseline (~88% on IMDB); no embedding needed |
| **Bidirectional LSTM** | Captures sequential dependencies in both directions; ~87-90% |
| **Conv1D** | Captures local n-gram patterns efficiently; ~88-91% |

### Architecture Decision Guide

```
Small dataset / fast inference?     →  Dense + TF-IDF
Long sequences, long-range context? →  LSTM / Bidirectional LSTM
Speed + local patterns matter?      →  Conv1D
State-of-the-art NLP (2024)?        →  BERT / RoBERTa / DistilBERT
```

### What's Next?
- 🔤 **Pre-trained embeddings** — GloVe, FastText (use them instead of random init)
- 🤖 **Transfer Learning** — fine-tune BERT on your own sentiment dataset
- 🌐 **Multi-class classification** — extend to 5-star rating prediction
- 📊 **Attention mechanisms** — give the model "where to look" in long reviews

---
*TCHAPGA TCHITO C. | University of Buea — COT | 2025/2026*